# 08 · Lista de casa: o treino completo

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

## Instruções

- Trabalhe em uma cópia deste notebook e substitua cada `...` pelo seu código.
- A primeira célula reúne os dados e as funções da aula (`ImagensDeRoupas`, `treinar_uma_epoca`, `avaliar`, `ajustar`). Leia antes de começar.
- Cerca de metade da nota está nas respostas escritas. Use os números que você obteve.
- O conjunto de teste só aparece na questão 4.
- Antes de entregar, use **Reiniciar sessão e executar tudo** (alguns minutos em CPU; menos com GPU).

| Questão | Tema | Pontos |
|---|---|---|
| 1 | Um `Dataset` com ruído | 2,0 |
| 2 | Tamanho do lote | 2,0 |
| 3 | Três experimentos, registrados | 3,0 |
| 4 | Análise de erros no teste | 2,0 |
| 5 | *Checkpoints* | 1,0 |

In [ ]:
import gzip
import math
import random
import time
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEMENTE = 42
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "· dispositivo:", dispositivo)

# ---------- dados ----------
BASE = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/b2617bb6d3ffa2e429640350f613e3291e10b141/data/fashion/"
PASTA = Path("fashion_mnist")
PASTA.mkdir(exist_ok=True)
for arquivo in ["train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz", "t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz"]:
    if not (PASTA / arquivo).exists():
        urlretrieve(BASE + arquivo, PASTA / arquivo)


def ler_imagens(caminho):
    with gzip.open(caminho) as f:
        return np.frombuffer(f.read(), dtype=np.uint8, offset=16).reshape(-1, 28, 28)


def ler_rotulos(caminho):
    with gzip.open(caminho) as f:
        return np.frombuffer(f.read(), dtype=np.uint8, offset=8)


imagens, rotulos = ler_imagens(PASTA / "train-images-idx3-ubyte.gz"), ler_rotulos(PASTA / "train-labels-idx1-ubyte.gz")
imagens_teste, rotulos_teste = ler_imagens(PASTA / "t10k-images-idx3-ubyte.gz"), ler_rotulos(PASTA / "t10k-labels-idx1-ubyte.gz")
NOMES = ["Camiseta", "Calça", "Suéter", "Vestido", "Casaco", "Sandália", "Camisa", "Tênis", "Bolsa", "Bota"]

ordem = torch.randperm(60_000, generator=torch.Generator().manual_seed(SEMENTE)).numpy()
ind_ajuste, ind_valid = ordem[:50_000], ordem[50_000:]
MEDIA, DESVIO = float(imagens[ind_ajuste].mean() / 255), float(imagens[ind_ajuste].std() / 255)


class ImagensDeRoupas(Dataset):
    def __init__(self, imagens, rotulos):
        x = torch.from_numpy(imagens.copy()).float().div(255)
        self.x = ((x - MEDIA) / DESVIO).unsqueeze(1)
        self.y = torch.from_numpy(rotulos.copy()).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.x[i], self.y[i]


dados_ajuste = ImagensDeRoupas(imagens[ind_ajuste], rotulos[ind_ajuste])
dados_valid = ImagensDeRoupas(imagens[ind_valid], rotulos[ind_valid])
dados_teste = ImagensDeRoupas(imagens_teste, rotulos_teste)
carregador_ajuste = DataLoader(dados_ajuste, batch_size=128, shuffle=True)
carregador_valid = DataLoader(dados_valid, batch_size=1000)
carregador_teste = DataLoader(dados_teste, batch_size=1000)


# ---------- funções de treino da aula ----------
def semear(semente=SEMENTE):
    random.seed(semente); np.random.seed(semente); torch.manual_seed(semente)


def contar_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters())


def treinar_uma_epoca(modelo, carregador, funcao_perda, otimizador):
    modelo.train()
    soma_perda, acertos, total = 0.0, 0, 0
    for xb, yb in carregador:
        xb, yb = xb.to(dispositivo), yb.to(dispositivo)
        otimizador.zero_grad()
        logits = modelo(xb)
        perda = funcao_perda(logits, yb)
        perda.backward()
        otimizador.step()
        soma_perda += perda.item() * len(yb)
        acertos += (logits.argmax(dim=1) == yb).sum().item()
        total += len(yb)
    return soma_perda / total, acertos / total


@torch.no_grad()
def avaliar(modelo, carregador, funcao_perda):
    modelo.eval()
    soma_perda, acertos, total = 0.0, 0, 0
    for xb, yb in carregador:
        xb, yb = xb.to(dispositivo), yb.to(dispositivo)
        logits = modelo(xb)
        soma_perda += funcao_perda(logits, yb).item() * len(yb)
        acertos += (logits.argmax(dim=1) == yb).sum().item()
        total += len(yb)
    return soma_perda / total, acertos / total


PASTA_MODELOS = Path("modelos")
PASTA_MODELOS.mkdir(exist_ok=True)
experimentos, modelos = {}, {}


def ajustar(modelo, nome, epocas=6, taxa=1e-3, weight_decay=0.0, paciencia=3, carregador=None, descricao=""):
    """Treina, salva o melhor checkpoint em modelos/<nome>.pt, recarrega-o e registra o experimento."""
    carregador = carregador or carregador_ajuste
    modelo.to(dispositivo)
    funcao_perda = nn.CrossEntropyLoss()
    otimizador = torch.optim.AdamW(modelo.parameters(), lr=taxa, weight_decay=weight_decay)
    caminho = PASTA_MODELOS / f"{nome}.pt"
    historico, melhor_perda, sem_melhora = [], float("inf"), 0
    inicio = time.perf_counter()
    for epoca in range(1, epocas + 1):
        perda_ajuste, acuracia_ajuste = treinar_uma_epoca(modelo, carregador, funcao_perda, otimizador)
        perda_valid, acuracia_valid = avaliar(modelo, carregador_valid, funcao_perda)
        historico.append({"epoca": epoca, "perda_ajuste": perda_ajuste, "perda_valid": perda_valid,
                          "acuracia_ajuste": acuracia_ajuste, "acuracia_valid": acuracia_valid})
        if perda_valid < melhor_perda:
            melhor_perda, sem_melhora = perda_valid, 0
            torch.save({"modelo": modelo.state_dict(), "otimizador": otimizador.state_dict(), "epoca": epoca,
                        "perda_valid": perda_valid, "acuracia_valid": acuracia_valid}, caminho)
        else:
            sem_melhora += 1
        if sem_melhora >= paciencia:
            break
    ponto = torch.load(caminho, map_location=dispositivo, weights_only=True)
    modelo.load_state_dict(ponto["modelo"])
    modelos[nome] = modelo
    experimentos[nome] = {"descrição": descricao, "parâmetros": contar_parametros(modelo), "épocas": epoca, "melhor época": ponto["epoca"],
                          "perda valid": ponto["perda_valid"], "acurácia valid": ponto["acuracia_valid"],
                          "segundos": round(time.perf_counter() - inicio, 1)}
    print(f"[{nome}] {epoca} épocas · melhor: época {ponto['epoca']} · acurácia de validação {ponto['acuracia_valid']:.4f} · {experimentos[nome]['segundos']} s")
    return pd.DataFrame(historico)


class RedeConvolucional(nn.Module):
    def __init__(self, p_dropout=0.0):
        super().__init__()
        self.extrator = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classificador = nn.Sequential(nn.Flatten(), nn.Dropout(p_dropout), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        return self.classificador(self.extrator(x))


def criar_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))


print(f"ajuste {len(dados_ajuste)} · validação {len(dados_valid)} · teste {len(dados_teste)}")

## Questão 1 (2,0 pontos): um `Dataset` com ruído

Somar ruído às imagens durante o treino é outra forma de aumento de dados. Escreva a classe `ImagensComRuido`:

- o construtor recebe `imagens`, `rotulos`, `desvio_ruido` (padrão 0,3) e `treino` (padrão `True`), e prepara as imagens como a classe `ImagensDeRoupas` (escala, padronização com `MEDIA` e `DESVIO`, eixo do canal);
- `__getitem__` devolve a imagem e o rótulo; quando `treino` é `True`, soma à imagem um ruído normal de média zero e desvio padrão `desvio_ruido`, **sorteado a cada acesso** (`torch.randn_like`); quando é `False`, devolve a imagem limpa.

In [ ]:
# ✏️ Questão 1
class ImagensComRuido(Dataset):
    # SEU CÓDIGO AQUI
    def __init__(self, imagens, rotulos, desvio_ruido=0.3, treino=True):
        ...
    
    def __len__(self):
        ...
    
    def __getitem__(self, i):
        ...

In [ ]:
# ✅ Conferência da questão 1 (não altere)
_com = ImagensComRuido(imagens[ind_valid][:500], rotulos[ind_valid][:500])
_sem = ImagensComRuido(imagens[ind_valid][:500], rotulos[ind_valid][:500], treino=False)
assert len(_com) == 500
_a, _ya = _com[3]; _b, _ = _com[3]; _c, _yc = _sem[3]; _d, _ = _sem[3]
assert tuple(_a.shape) == (1, 28, 28) and _a.dtype == torch.float32 and _ya.dtype == torch.int64 and int(_ya) == int(rotulos[ind_valid][3])
assert not torch.equal(_a, _b), "em modo de treino, dois acessos ao mesmo índice devem dar imagens diferentes"
assert torch.equal(_c, _d) and torch.allclose(_c, dados_valid[3][0]), "em modo de avaliação, a imagem deve ser a mesma de ImagensDeRoupas"
assert 0.25 < (_a - _c).std().item() < 0.35, "o desvio padrão do ruído deveria estar perto de 0,3"
assert torch.equal(_sem[3][0], _c), "o acesso com ruído não pode alterar a imagem guardada"

fig, eixos = plt.subplots(1, 3, figsize=(6, 2.3))
for ax, (figura, titulo) in zip(eixos, [(_c, "limpa"), (_a, "com ruído"), (_b, "com ruído, outro acesso")]):
    ax.imshow(figura[0], cmap="gray_r"); ax.axis("off"); ax.set_title(titulo, fontsize=9)
print("Formato correto.")

**1b)** Por que o ruído é aplicado só em modo de treino? Que outra técnica vista em aula tem um comportamento em treino e outro em avaliação, e o que as duas têm em comum? Por que o ruído é sorteado dentro do `__getitem__`, e não uma vez no construtor?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 2 (2,0 pontos): tamanho do lote

Para cada tamanho de lote em 32, 128 e 1024: fixe as sementes com `semear()`, crie um `DataLoader` de ajuste com embaralhamento, crie um MLP novo com `criar_mlp()` (no dispositivo), e treine por **3 épocas** com `treinar_uma_epoca`, usando `nn.CrossEntropyLoss` e `torch.optim.Adam` com `lr=1e-3`. Meça o tempo total das três épocas e, ao final, a acurácia de validação.

Monte `tabela_q2`, um DataFrame com índice `lote` e colunas `passos_por_epoca`, `segundos` e `acuracia_valid`.

In [ ]:
# ✏️ Questão 2
# SEU CÓDIGO AQUI
tabela_q2 = ...

In [ ]:
# ✅ Conferência da questão 2 (não altere)
assert list(tabela_q2.index) == [32, 128, 1024] and list(tabela_q2.columns) == ["passos_por_epoca", "segundos", "acuracia_valid"]
assert tabela_q2["passos_por_epoca"].tolist() == [math.ceil(50_000 / lote) for lote in [32, 128, 1024]], "passos por época = número de lotes do DataLoader"
assert (tabela_q2["segundos"] > 0).all() and tabela_q2["acuracia_valid"].between(0.75, 0.95).all()
print("Formato correto.")

**2b)** Compare os três tamanhos de lote em tempo, número de atualizações e acurácia depois de três épocas. Por que o lote de 1024 termina com acurácia menor, se viu exatamente as mesmas imagens? O que você mudaria para usá-lo bem? Que limite físico existe para o tamanho do lote?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 3 (3,0 pontos): três experimentos, registrados

Rode **pelo menos três** experimentos com a função `ajustar`, com **no máximo 6 épocas** cada, procurando a melhor acurácia de validação.

- Os nomes devem começar com `q3_` (por exemplo `q3_base`, `q3_dropout`).
- O primeiro deve ser a referência: `ajustar(RedeConvolucional(), "q3_base", descricao="referência")`.
- Cada experimento seguinte deve mudar **uma coisa** em relação a um anterior. Use o argumento `descricao` para registrar a mudança.
- Chame `semear()` antes de criar cada modelo.

Ao final, monte `tabela_q3` (a tabela de experimentos, só com as linhas `q3_`) e guarde em `melhor_q3` o nome do experimento escolhido.

In [ ]:
# ✏️ Questão 3
# SEU CÓDIGO AQUI
...
tabela_q3 = ...
melhor_q3 = ...

In [ ]:
# ✅ Conferência da questão 3 (não altere)
assert len(tabela_q3) >= 3 and all(nome.startswith("q3_") for nome in tabela_q3.index), "são pedidos pelo menos três experimentos com nomes iniciados por q3_"
assert "q3_base" in tabela_q3.index and (tabela_q3["épocas"].astype(int) <= 6).all(), "inclua a referência e respeite o limite de 6 épocas"
assert (tabela_q3["descrição"].str.len() > 0).all(), "descreva cada experimento no argumento descricao"
assert melhor_q3 in tabela_q3.index and melhor_q3 in modelos
assert float(tabela_q3["acurácia valid"].astype(float).max()) > 0.89
print(f"Formato correto: {len(tabela_q3)} experimentos; escolhido {melhor_q3}, com {float(tabela_q3.loc[melhor_q3, 'acurácia valid']):.4f} de acurácia de validação.")

**3b)** Para cada experimento além da referência: o que você mudou, o que esperava que acontecesse e o que aconteceu?

**3c)** Qual experimento você escolheu, e com que critério? As diferenças entre os experimentos são grandes o bastante para confiar nelas? O que você faria para ter mais segurança, e o que tentaria em seguida se tivesse mais tempo?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 4 (2,0 pontos): análise de erros no teste

Com o modelo escolhido (`modelos[melhor_q3]`), agora no **conjunto de teste**:

a) Calcule `previstos_q4` (array com a classe prevista para cada imagem de teste) e `matriz_q4` (matriz de confusão 10 × 10, com as classes verdadeiras nas linhas).

b) Monte `acertos_q4`: uma `Series` indexada pelo nome da classe com a proporção de acertos de cada classe, em ordem crescente.

c) Monte `top3_q4`: uma lista com as três confusões mais frequentes, cada uma como tupla `(nome da classe verdadeira, nome da classe prevista, quantidade)`, da mais para a menos frequente.

d) Mostre 12 imagens do erro mais frequente (as que pertencem à primeira classe de `top3_q4[0]` e foram previstas como a segunda).

In [ ]:
# ✏️ Questão 4
# SEU CÓDIGO AQUI
previstos_q4 = ...
matriz_q4 = ...
acertos_q4 = ...
top3_q4 = ...

In [ ]:
# ✅ Conferência da questão 4 (não altere)
assert previstos_q4.shape == (10_000,) and matriz_q4.shape == (10, 10) and matriz_q4.sum() == 10_000
assert (matriz_q4.sum(axis=1) == 1000).all(), "as linhas devem ser as classes verdadeiras (1000 imagens cada)"
assert acertos_q4.is_monotonic_increasing and np.isclose(acertos_q4[NOMES[0]], matriz_q4[0, 0] / 1000)
_m = matriz_q4 - np.diag(matriz_q4.diagonal())
assert len(top3_q4) == 3 and top3_q4[0][2] == _m.max() and [t[2] for t in top3_q4] == sorted((t[2] for t in top3_q4), reverse=True)
assert all(matriz_q4[NOMES.index(v), NOMES.index(p)] == q and v != p for v, p, q in top3_q4), "as quantidades não correspondem à matriz"
print(f"Formato correto. Acurácia no teste: {matriz_q4.diagonal().sum() / 10_000:.2%}")

**4e)** Olhando as 12 imagens do erro mais frequente: você, sem ver o rótulo, acertaria? O que isso sugere sobre o teto de desempenho neste conjunto?

**4f)** Proponha uma ação concreta para reduzir esse tipo de erro que **não** seja "usar uma rede maior". Depois responda: tendo visto esses resultados no teste, por que seria um erro voltar à questão 3, ajustar o modelo e medir de novo no mesmo teste?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 5 (1,0 ponto): *checkpoints*

a) O que precisa estar em um *checkpoint* para **usar** o modelo depois? E para **retomar o treino** de onde parou? Explique a diferença.

b) Por que a função `ajustar` recarrega o *checkpoint* ao final do treino, em vez de ficar com o modelo da última época?

c) Declare como usou assistentes de IA nesta lista: em quais questões, para quê, e o que precisou corrigir.

✏️ **Sua resposta:**

_Escreva aqui._